# WSMTE — 06_evaluation.ipynb
**LOCAL PC** | Day 5 Tasks 5.13–5.25

All metrics, statistical tests, and figure generation.

**Prerequisites:** All files downloaded from Kaggle sessions:
- `ablation/ablation_results_AG.csv`
- `ablation/ablation_results_H.csv`
- `results/saved_models/config_h_best.keras`
- `results/figures/shap_summary.png` (already generated by notebook 05)
- `data/processed/X_test.npy`, `y_clf_test.npy`, `y_reg_test.npy`
- `data/processed/final_dataset.csv`

In [1]:
# ── Cell 1: Imports + paths ────────────────────────────────────────────────
import sys, os
ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for local PC
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json
from scipy import stats

from config.config import CONFIG

# Ensure output directories exist
for d in [CONFIG['figures_dir'], CONFIG['tables_dir'], CONFIG['models_dir']]:
    os.makedirs(d, exist_ok=True)

print('Imports OK')
print(f"Figures dir: {os.path.join(ROOT,CONFIG['figures_dir'])}")
print(f"Tables dir : {os.path.join(ROOT, CONFIG['tables_dir'])}")

Imports OK
Figures dir: d:\WSMTE\results/figures/
Tables dir : d:\WSMTE\results/tables/


## Step 1 — Merge Ablation Results (A–H)

In [2]:
# ── Cell 2: Load + merge AG and H results ─────────────────────────────────
df_ag = pd.read_csv(os.path.join(ROOT, 'ablation/ablation_results_AG.csv'))
df_h  = pd.read_csv(os.path.join(ROOT, 'ablation/ablation_results_H.csv'))


print(f'Results A-G shape : {df_ag.shape}  configs: {df_ag["config"].unique().tolist()}')
print(f'Results H shape   : {df_h.shape}  configs: {df_h["config"].unique().tolist()}')

ablation = pd.concat([df_ag, df_h], ignore_index=True)
ablation.to_csv(os.path.join(ROOT, CONFIG['ablation_results']), index=False)

print(f'\nMerged -> {CONFIG["ablation_results"]}  shape: {ablation.shape}')
print(ablation.groupby('config')['accuracy'].count().rename('n_runs').to_string())

Results A-G shape : (70, 12)  configs: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Results H shape   : (30, 12)  configs: ['H']

Merged -> ablation/ablation_results.csv  shape: (100, 12)
config
A    10
B    10
C    10
D    10
E    10
F     0
G    10
H    30


## Step 2 — Per-Config Summary Statistics

In [3]:
# ── Cell 3: Mean / Max / Std per config ───────────────────────────────────
summary_cols = ['accuracy', 'balanced_accuracy', 'auc', 'precision', 'recall', 'f1']
reg_cols = ['rmse', 'mae', 'r2']

summary = ablation.groupby('config').agg(
    mean_acc=('accuracy', 'mean'),
    max_acc=('accuracy', 'max'),
    std_acc=('accuracy', 'std'),
    mean_bacc=('balanced_accuracy', 'mean'),
    mean_auc=('auc', 'mean'),
    mean_f1=('f1', 'mean'),
    mean_rmse=('rmse', 'mean'),
    n_runs=('run', 'count'),
).round(4)
print(summary.to_string())

summary.to_csv(os.path.join(ROOT,CONFIG['tables_dir']) + 'ablation_summary.csv')
print(f"\nSaved -> {os.path.join(ROOT, CONFIG['tables_dir'], 'ablation_summary.csv')}")

        mean_acc  max_acc  std_acc  mean_bacc  mean_auc  mean_f1  mean_rmse  n_runs
config                                                                             
A         0.5654   0.5769   0.0365     0.5017    0.5016   0.6835        NaN      10
B         0.5737   0.5769   0.0069     0.4980    0.4923   0.7281        NaN      10
C         0.5782   0.5833   0.0027     0.5015    0.5021   0.7323        NaN      10
D         0.5776   0.5833   0.0020     0.5008    0.5098   0.7320        NaN      10
E         0.5750   0.5769   0.0061     0.4987    0.5021   0.7297        NaN      10
F            NaN      NaN      NaN        NaN       NaN      NaN     0.2620      10
G         0.5083   0.5897   0.0425     0.5341    0.5527   0.4322     0.2591      10
H         0.5541   0.6538   0.0620     0.5664    0.6072   0.5176     0.1911      30

Saved -> d:\WSMTE\results/tables/ablation_summary.csv


## Step 3 — Wilcoxon Signed-Rank Test: Config B vs Config H

In [4]:
# ── Cell 4: Wilcoxon test ──────────────────────────────────────────────────
acc_b = ablation[ablation['config'] == 'B']['accuracy'].values
acc_h = ablation[ablation['config'] == 'H']['accuracy'].values

# Use min(len) for paired test (B=10, H=30 -> compare first 10 pairs)
n_pairs = min(len(acc_b), len(acc_h))
stat, p_value = stats.wilcoxon(acc_b[:n_pairs], acc_h[:n_pairs])

print(f'Wilcoxon signed-rank test: Config B vs Config H')
print(f'  n pairs     : {n_pairs}')
print(f'  Config B    : mean={acc_b.mean():.4f}  std={acc_b.std():.4f}')
print(f'  Config H    : mean={acc_h.mean():.4f}  std={acc_h.std():.4f}')
print(f'  W statistic : {stat:.4f}')
print(f'  p-value     : {p_value:.6f}')
print(f'  Significant : {"YES" if p_value < 0.05 else "NO"} (alpha=0.05)')

wilcoxon_result = {
    'test': 'Wilcoxon signed-rank',
    'config_a': 'B', 'config_b': 'H',
    'n_pairs': n_pairs,
    'mean_B': round(float(acc_b.mean()), 4),
    'mean_H': round(float(acc_h.mean()), 4),
    'W_statistic': round(float(stat), 4),
    'p_value': round(float(p_value), 6),
    'significant_at_0_05': bool(p_value < 0.05),
}
with open(os.path.join(ROOT,CONFIG['results_dir']) + 'wilcoxon_result.json', 'w') as f:
    json.dump(wilcoxon_result, f, indent=2)
print(f"Saved -> {os.path.join(ROOT, CONFIG['results_dir'], 'wilcoxon_result.json')}")

Wilcoxon signed-rank test: Config B vs Config H
  n pairs     : 10
  Config B    : mean=0.5737  std=0.0066
  Config H    : mean=0.5541  std=0.0610
  W statistic : 14.5000
  p-value     : 0.205078
  Significant : NO (alpha=0.05)
Saved -> d:\WSMTE\results/wilcoxon_result.json


## Step 4 — Ablation Comparison Bar Chart

In [5]:
# ── Cell 5: ablation_comparison.png ───────────────────────────────────────
config_order = ['A', 'B', 'C', 'D', 'E', 'G', 'H']
means = [summary.loc[c, 'mean_acc'] if c in summary.index else 0 for c in config_order]
stds  = [summary.loc[c, 'std_acc']  if c in summary.index else 0 for c in config_order]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(config_order, means, yerr=stds, capsize=5,
              color=['#4C72B0','#55A868','#C44E52','#8172B2',
                     '#CCB974','#64B5CD','#E8735A'],
              alpha=0.85, edgecolor='black', linewidth=0.6)

# Kotekar baseline
ax.axhline(0.5853, color='red', linestyle='--', linewidth=1.2,
           label='Kotekar baseline (0.5853)')

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{m:.4f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Ablation Config')
ax.set_ylabel('Mean Test Accuracy')
ax.set_title('WSMTE Ablation Study — Mean Accuracy per Config (±1 std)')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(ROOT,CONFIG['figures_dir']) + 'ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(ROOT, CONFIG['figures_dir'], 'ablation_comparison.png')}")

Saved -> d:\WSMTE\results/figures/ablation_comparison.png


C:\Users\seera\AppData\Local\Temp\ipykernel_32936\3973024771.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 5 — Load Best Config H Model + Test Predictions

In [6]:
# ── Cell 6: Load model + test data ────────────────────────────────────────
import tensorflow as tf
from src.models.wsmte import WSMTEModel, build_wsmte, build_wsmte_pso
from src.models.losses import uncertainty_weighted_loss


X_test      = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'X_test.npy'))
y_clf_test  = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'y_clf_test.npy'))
y_reg_test  = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'y_reg_test.npy'))


print(f'X_test shape     : {X_test.shape}')
print(f'y_clf_test shape : {y_clf_test.shape}  labels={set(y_clf_test.tolist())}')
print(f'y_reg_test shape : {y_reg_test.shape}')

# Load PSO weights and rebuild model — avoids WSMTEModel serialization issues
with open(os.path.join(ROOT, 'results', 'pso_weights.json')) as f:
    pso_info = json.load(f)
pso_w = np.array([pso_info['w_lstm'], pso_info['w_tcn'], pso_info['w_gru']])

h_model = build_wsmte_pso(CONFIG, pso_w)
h_model.compile(optimizer=tf.keras.optimizers.Adam(CONFIG['learning_rate']))
h_model.load_weights(os.path.join(ROOT, CONFIG['models_dir'], 'config_h_best.keras'))
print(f'Config H model loaded OK. Trainable params: {h_model.count_params():,}')



raw_preds = h_model(X_test, training=False)
y_reg_pred  = raw_preds[0].numpy().ravel()
y_clf_proba = raw_preds[1].numpy().ravel()
y_clf_pred  = (y_clf_proba >= 0.5).astype(int)

print(f'clf_proba range : {y_clf_proba.min():.4f} to {y_clf_proba.max():.4f}')
print(f'reg_pred  range : {y_reg_pred.min():.6f} to {y_reg_pred.max():.6f}')

X_test shape     : (156, 5, 9)
y_clf_test shape : (156,)  labels={0, 1}
y_reg_test shape : (156,)

Config H model loaded OK. Trainable params: 82,754


d:\WSMTE\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 66 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


clf_proba range : 0.0930 to 0.9091
reg_pred  range : 0.864786 to 1.177138


## Step 6 — Confusion Matrix

In [7]:
# ── Cell 7: confusion_matrix.png ──────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_clf_test, y_clf_pred)
print('Confusion Matrix (rows=actual, cols=predicted):')
print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
print(f'  FN={cm[1,0]}  TP={cm[1,1]}')

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Down (0)', 'Up (1)']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Config H — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(os.path.join(ROOT,CONFIG['figures_dir']) + 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(ROOT, CONFIG['figures_dir'], 'confusion_matrix.png')}")

Confusion Matrix (rows=actual, cols=predicted):
  TN=30  FP=36
  FN=18  TP=72
Saved -> d:\WSMTE\results/figures/confusion_matrix.png


C:\Users\seera\AppData\Local\Temp\ipykernel_32936\3459779862.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 7 — AUC-ROC Curve

In [8]:
# ── Cell 8: auc_roc_curve.png ─────────────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_clf_test, y_clf_proba)
auc_score = roc_auc_score(y_clf_test, y_clf_proba)
print(f'AUC-ROC: {auc_score:.4f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=2, color='#4C72B0', label=f'Config H (AUC = {auc_score:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Config H (Test Set)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, CONFIG['figures_dir']) + 'auc_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(ROOT, CONFIG['figures_dir'], 'auc_roc_curve.png')}")

AUC-ROC: 0.6215
Saved -> d:\WSMTE\results/figures/auc_roc_curve.png


C:\Users\seera\AppData\Local\Temp\ipykernel_32936\1696725996.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ── Cell 8b: SHAP analysis (re-run locally with all 9 features) ───────────
from src.evaluation.shap_analysis import run_shap_analysis

X_train_shap = np.load(os.path.join(ROOT,CONFIG['processed_data_dir']) + 'X_train.npy')

shap_values = run_shap_analysis(
    model=h_model,
    X_test=X_test,
    feature_names=CONFIG['feature_columns'],   # all 9 features
    X_train=X_train_shap,                      # background from training data
    save_path=os.path.join(ROOT,CONFIG['figures_dir']) + 'shap_summary.png',
    background_size=100,
    max_display=9,
)
print(f"SHAP values shape : {shap_values[0].shape if isinstance(shap_values, list) else shap_values.shape}")


d:\WSMTE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running SHAP GradientExplainer...


d:\WSMTE\.venv\Lib\site-packages\keras\src\models\functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input
Received: inputs=['Tensor(shape=(156, 5, 9))']
  warnings.warn(msg)
d:\WSMTE\.venv\Lib\site-packages\keras\src\models\functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input
Received: inputs=['Tensor(shape=(50, 5, 9))']
  warnings.warn(msg)
d:\WSMTE\src\evaluation\shap_analysis.py:77: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


SHAP summary saved → d:\WSMTE\results/figures/shap_summary.png
SHAP values shape : (156, 5, 9, 1)


d:\WSMTE\src\evaluation\shap_analysis.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 8 — Training Loss Curves
> Uses per-run history from the best Config H run (reconstructed from ablation CSV).

In [10]:
# ── Cell 9: loss_curves.png ───────────────────────────────────────────────
# Re-train best Config H run to capture history (or load if saved separately)
# Here we do a quick 1-run retrain with the best seed to get clean curves.
from src.models.wsmte import build_wsmte, build_wsmte_pso
from src.training.callbacks import get_callbacks

# Load best PSO weights
with open(os.path.join(ROOT, 'results', 'pso_weights.json')) as f:
    pso_info = json.load(f)
pso_w = np.array([pso_info['w_lstm'], pso_info['w_tcn'], pso_info['w_gru']])
print(f'PSO weights: LSTM={pso_w[0]:.4f}, TCN={pso_w[1]:.4f}, GRU={pso_w[2]:.4f}')

# Load best seed from results
best_seed = int(ablation[ablation['config']=='H']['accuracy'].idxmax())
h_df = ablation[ablation['config'] == 'H']
best_seed_val = int(h_df.loc[h_df['accuracy'].idxmax(), 'seed'])
tf.random.set_seed(best_seed_val)
np.random.seed(best_seed_val)

X_train = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'X_train.npy'))
X_val   = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'X_val.npy'))
y_reg_train = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'y_reg_train.npy'))
y_clf_train = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'y_clf_train.npy'))
y_reg_val   = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'y_reg_val.npy'))
y_clf_val   = np.load(os.path.join(ROOT, CONFIG['processed_data_dir'], 'y_clf_val.npy'))

lc_model = build_wsmte_pso(CONFIG, pso_w)
lc_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate'])
)
history = lc_model.fit(
    X_train, [y_reg_train, y_clf_train],
    validation_data=(X_val, [y_reg_val, y_clf_val]),
    epochs=CONFIG['pso_finetune_epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=get_callbacks(CONFIG, run_id='loss_curve_run'),
    verbose=0,
)
print(f'Loss curve training done. Epochs run: {len(history.history["loss"])}')

PSO weights: LSTM=0.2636, TCN=0.2142, GRU=0.5223
Restoring model weights from the end of the best epoch: 20.
Loss curve training done. Epochs run: 20


In [11]:
# ── Cell 10: Plot loss curves ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
epochs = range(1, len(history.history['loss']) + 1)
ax.plot(epochs, history.history['loss'],     label='Train loss', lw=2)
ax.plot(epochs, history.history['val_loss'], label='Val loss',   lw=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MTL uncertainty-weighted)')
ax.set_title('Config H — Training vs Validation Loss')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, CONFIG['figures_dir']) + 'loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(ROOT, CONFIG['figures_dir'], 'loss_curves.png')}")

Saved -> d:\WSMTE\results/figures/loss_curves.png


C:\Users\seera\AppData\Local\Temp\ipykernel_32936\2995377551.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 9 — Granger Causality Tests

In [ ]:
# ── Cell 11: Granger causality (article-level features + market, lags 1-10) ─
from src.evaluation.granger_test import run_granger_tests

# Load final_dataset.csv — has raw Close and polarity_market columns
final_df = pd.read_csv(os.path.join(ROOT, CONFIG['processed_data_dir'] + 'final_dataset.csv'))
final_df = final_df.sort_values('date').reset_index(drop=True)

print(f'final_dataset shape: {final_df.shape}')
print(f'Columns: {list(final_df.columns)}')

# Load kotekar_sentiment.csv — article-level, one row per article
kotekar_art = pd.read_csv(os.path.join(ROOT, CONFIG['kotekar_sentiment_file']))
kotekar_art['date'] = pd.to_datetime(kotekar_art['date']).dt.date
print(f'kotekar_sentiment shape: {kotekar_art.shape}')
print(f'kotekar_sentiment columns: {list(kotekar_art.columns)}')

granger_df = run_granger_tests(
    df=final_df,
    kotekar_art=kotekar_art,
    max_lag=CONFIG['granger_max_lag'],     # 10
    save_path=os.path.join(ROOT, CONFIG['tables_dir'], 'granger_results.csv'),
)
print('\n')
print(granger_df.to_string())

## Step 10 — Trading Simulation (Long-Only, Kotekar Algorithm 1)

In [13]:
# ── Cell 12: Trading simulation ───────────────────────────────────────────
from src.evaluation.trading_sim import run_trading_simulation

# Actual next-day log returns for test period
# final_dataset rows correspond to test set (after split)
n_total = len(final_df)
test_start_idx = int(n_total * (CONFIG['train_ratio']) + CONFIG['val_ratio'])
test_df = final_df.iloc[test_start_idx:].reset_index(drop=True)

# Day-over-day returns for test set (aligned to predictions)
actual_returns = test_df['Close'].pct_change().fillna(0).values
# Trim to exactly match prediction count (156 windows)
actual_returns = actual_returns[-len(y_clf_proba):]


print(f'Test predictions shape: {y_clf_proba.shape}')
print(f'Actual returns shape  : {actual_returns.shape}')

sim_results = run_trading_simulation(
    y_pred_proba=y_clf_proba,
    actual_returns=actual_returns,
    risk_free_rate=CONFIG['risk_free_rate'],
   save_path=os.path.join(ROOT, CONFIG['figures_dir'], 'trading_simulation.png'))


print('\nTrading simulation results:')
for k, v in sim_results.items():
    if isinstance(v, float):
        print(f'  {k:25s}: {v:.4f}')
    else:
        print(f'  {k:25s}: {v}')

Test predictions shape: (156,)
Actual returns shape  : (156,)

── Trading Simulation Summary ──────────────────────
  Strategy total return : 25.75%
  Buy & Hold return     : 15.47%
  Strategy Sharpe ratio : 3.2580
  Buy & Hold Sharpe     : 1.5988
  Long days             : 108
  Flat days             : 48
────────────────────────────────────────────────────
Trading simulation plot saved → d:\WSMTE\results/figures/trading_simulation.png

Trading simulation results:
  strategy_total_return    : 0.2575
  buyhold_total_return     : 0.1547
  strategy_sharpe          : 3.2580
  buyhold_sharpe           : 1.5988
  n_long_days              : 108
  n_flat_days              : 48
  strategy_cum_returns     : [ 0.0090968   0.01532363  0.01532363  0.01532363  0.01532363  0.01532363
  0.01532363  0.01292867  0.01292867 -0.00059424 -0.00059424 -0.01443907
 -0.00450885 -0.00450885 -0.00450885 -0.00450885  0.00304546  0.00304546
  0.01249405  0.0122306   0.0122306   0.0122306   0.01379884  0.01474875
 

In [14]:
# ── Cell 13: Save trading results table ───────────────────────────────────
trading_row = {k: v for k, v in sim_results.items() if not hasattr(v, '__len__')}
pd.DataFrame([trading_row]).to_csv(
    os.path.join(ROOT,CONFIG['tables_dir'] )+ 'trading_results.csv', index=False)
print(f"Saved -> {os.path.join(ROOT, CONFIG['tables_dir'], 'trading_results.csv')}")

Saved -> d:\WSMTE\results/tables/trading_results.csv


## Step 11 — Optional: Wavelet Denoising Visualisation

In [15]:
# ── Cell 14: wavelet_denoising.png (optional) ─────────────────────────────
import pywt
from src.data.preprocessor import coif3_denoise

final_df2 = pd.read_csv(os.path.join(ROOT, CONFIG['processed_data_dir'], 'final_dataset.csv'))
raw_close = final_df2['Close_d']  # denoised is in final_dataset
# We need the original close — load merged_data
merged = pd.read_csv(os.path.join(ROOT, CONFIG['processed_data_dir'], 'merged_data.csv'))
merged = merged.dropna(subset=['Close']).reset_index(drop=True)

raw_prices = merged['Close'].values[:200].copy()
denoised   = coif3_denoise(raw_prices, CONFIG)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw_prices, alpha=0.7, label='Raw Close', lw=1)
ax.plot(denoised,   alpha=0.9, label='Coif3 Denoised', lw=1.5)
ax.set_xlabel('Trading Day')
ax.set_ylabel('Nifty 50 Index')
ax.set_title('Coif3 Wavelet Denoising — Close Price (first 200 days)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, CONFIG['figures_dir'], 'wavelet_denoising.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {os.path.join(ROOT, CONFIG['figures_dir'], 'wavelet_denoising.png')}")

Saved -> d:\WSMTE\results/figures/wavelet_denoising.png


C:\Users\seera\AppData\Local\Temp\ipykernel_32936\3597311398.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final Summary

In [16]:
# ── Cell 15: Summary of all outputs ──────────────────────────────────────
figures = [
    'loss_curves.png', 'confusion_matrix.png', 'auc_roc_curve.png',
    'ablation_comparison.png', 'trading_simulation.png',
    'shap_summary.png', 'wavelet_denoising.png',
]
tables = [
    'ablation_summary.csv', 'granger_results.csv', 'trading_results.csv'
]

print('=' * 55)
print('NOTEBOOK 06 COMPLETE')
print('=' * 55)
print('FIGURES:')
for fn in figures:
    path = CONFIG['figures_dir'] + fn
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {fn:35s}  {status}')

print('\nTABLES:')
for fn in tables:
    path = os.path.join(ROOT,CONFIG['tables_dir']) + fn
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {fn:35s}  {status}')

print(f'\nablation_results.csv: {os.path.join(ROOT, CONFIG["ablation_results"])}')
df_check = pd.read_csv(os.path.join(ROOT, CONFIG['ablation_results']))
print(df_check.groupby('config')['accuracy'].agg(['mean','max','std']).round(4).to_string())

NOTEBOOK 06 COMPLETE
FIGURES:
  loss_curves.png                      OK
  confusion_matrix.png                 OK
  auc_roc_curve.png                    OK
  ablation_comparison.png              OK
  trading_simulation.png               OK
  shap_summary.png                     MISSING
  wavelet_denoising.png                MISSING

TABLES:
  ablation_summary.csv                 OK
  granger_results.csv                  OK
  trading_results.csv                  OK

ablation_results.csv: d:\WSMTE\ablation/ablation_results.csv
          mean     max     std
config                        
A       0.5654  0.5769  0.0365
B       0.5737  0.5769  0.0069
C       0.5782  0.5833  0.0027
D       0.5776  0.5833  0.0020
E       0.5750  0.5769  0.0061
F          NaN     NaN     NaN
G       0.5083  0.5897  0.0425
H       0.5541  0.6538  0.0620
